In [ ]:
 !pip install fastapi uvicorn python-multipart \
              openai-whisper transformers torch torchvision \
              pillow opencv-python-headless ultralytics \
              pyngrok accelerate

In [ ]:
import whisper
import cv2
import torch
import numpy as np
from PIL import Image
from transformers import (
    BlipProcessor,
    BlipForConditionalGeneration,
    CLIPProcessor,
    CLIPModel,
)
from ultralytics import YOLO
from fastapi import FastAPI, File, UploadFile
from fastapi.middleware.cors import CORSMiddleware
import uvicorn
import threading
import tempfile
import os
import json
from pyngrok import ngrok

print("Imports OK")

In [ ]:
print("Loading Whisper (base)...")
whisper_model = whisper.load_model("base")       # swap to "small" for better accuracy

print("Loading BLIP...")
blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model     = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

print("Loading CLIP...")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model     = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")

print("Loading YOLOv8n...")
yolo_model = YOLO("yolov8n.pt")                  # auto-downloads on first run

print("✅ All models loaded and ready.")

In [ ]:
# ── CELL 4 : Inference helper functions ────────────────────

def extract_frames(video_path: str, every_n_seconds: int = 2):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    frames, count = [], 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if count % int(fps * every_n_seconds) == 0:
            frames.append(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
        count += 1
    cap.release()
    return frames


import subprocess
import tempfile
import os

def transcribe_audio(file_path):
    result = whisper_model.transcribe(file_path)
    return result["text"]


def describe_scene(frames: list) -> list:
    descriptions = []
    for frame in frames[:3]:
        inputs = blip_processor(frame, return_tensors="pt")
        out    = blip_model.generate(**inputs)
        desc   = blip_processor.decode(out[0], skip_special_tokens=True)
        descriptions.append(desc)
    return descriptions or ["no visual frames available"]


def classify_incident(frames: list) -> str:
    if not frames:
        return "medical emergency"
    labels  = ["road accident", "fire", "flood", "physical assault",
               "medical emergency", "normal scene"]
    results = []
    for frame in frames[:3]:
        inputs  = clip_processor(text=labels, images=frame,
                                 return_tensors="pt", padding=True)
        outputs = clip_model(**inputs)
        probs   = outputs.logits_per_image.softmax(dim=1)
        results.append(labels[probs.argmax().item()])
    return max(set(results), key=results.count)


def count_people(frames: list) -> int:
    max_count = 0
    for frame in frames[:3]:
        results   = yolo_model(np.array(frame))
        people    = sum(1 for r in results for box in r.boxes if int(box.cls) == 0)
        max_count = max(max_count, people)
    return max_count

In [ ]:
# ── CELL 5 : FastAPI app with single /infer endpoint ───────

app = FastAPI(title="WitnessChain Inference Server")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.post("/infer")
async def infer(file: UploadFile = File(...)):
    suffix = "." + file.filename.split(".")[-1] if file.filename and "." in file.filename else ".webm"
    with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as tmp:
        tmp.write(await file.read())
        tmp_path = tmp.name

    wav_path = tmp_path + "_audio.wav"
    try:
        is_video = suffix in [".webm", ".mp4", ".avi", ".mov", ".mkv"]

        # Convert to wav
        result = subprocess.run([
            "ffmpeg", "-y", "-i", tmp_path,
            "-ar", "16000", "-ac", "1", "-f", "wav", wav_path
        ], capture_output=True)

        if result.returncode != 0 or not os.path.exists(wav_path):
            # ffmpeg failed — skip transcription
            transcript = ""
        else:
            transcript = transcribe_audio(wav_path)

        frames             = extract_frames(tmp_path) if is_video else []
        scene_descriptions = describe_scene(frames)
        incident_type      = classify_incident(frames)
        people_count       = count_people(frames)
    finally:
        os.unlink(tmp_path)
        if os.path.exists(wav_path):
            os.unlink(wav_path)

    return {
        "transcript":         transcript,
        "scene_descriptions": scene_descriptions,
        "incident_type":      incident_type,
        "people_count":       people_count,
    }

@app.get("/ping")
def ping():
    return {"status": "colab-inference-server-running"}

In [ ]:
# ── CELL 6 : Start server + expose via ngrok ───────────────
# Sign up free at ngrok.com and paste your token below.

ngrok.set_auth_token("YOUR_NGROK_AUTH_TOKEN")  # paste your own token; never commit it

def _run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

server_thread = threading.Thread(target=_run_server, daemon=True)
server_thread.start()

public_url = ngrok.connect(8000)
print("=" * 60)
print(f"  COLAB INFER URL  →  {public_url}/infer")
print("  Copy this into VS Code  .env  as  COLAB_INFER_URL")
print("=" * 60)


In [ ]:
#import requests
#r = requests.post("http://localhost:8000/infer",
 #   files={"file": ("test.txt", b"hello", "text/plain")})
#print(r.json())